# ICA Stochastique — Démo et Comparaison

Ce notebook illustre et compare les quatre algorithmes ICA implémentés :

| Algorithme | Type | Référence |
|---|---|---|
| **Infomax (batch)** | Classique | Bell & Sejnowski (1995) |
| **FastICA (custom)** | Classique | Hyvärinen (1999) |
| **FastICA (sklearn)** | Référence | sklearn baseline |
| **SGD-ICA** | Stochastique | Mini-batch Infomax |
| **Adam-ICA** | Stochastique | Adam + Infomax |

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Génération des données synthétiques

On simule un problème BSS classique :
- **Sources indépendantes** : Laplace (super-gaussienne), sinusoïde (sous-gaussienne), uniforme
- **Mélange** via une matrice aléatoire A

In [ ]:
from experiments.benchmark import make_sources

S, A, X = make_sources(n_samples=3000, n_sources=3, source_type='mixed', random_state=42)

fig, axes = plt.subplots(3, 2, figsize=(12, 6))
t = np.arange(500)
labels_s = ['Laplace (super-G)', 'Sinusoïde (sous-G)', 'Uniforme (sous-G)']
colors = ['#e74c3c', '#2ecc71', '#3498db']

for i in range(3):
    axes[i, 0].plot(t, S[:500, i], color=colors[i], linewidth=0.9)
    axes[i, 0].set_ylabel(f's{i+1}', rotation=0, labelpad=12)
    axes[i, 0].set_title(labels_s[i] if i == 0 else '', fontsize=9)
    axes[i, 1].plot(t, X[:500, i], color='gray', linewidth=0.9, alpha=0.7)

axes[0, 0].set_title('Sources indépendantes S', fontweight='bold')
axes[0, 1].set_title('Mélanges observés X = SA^T', fontweight='bold')
plt.tight_layout()
plt.suptitle('Données synthétiques BSS', y=1.01, fontsize=13, fontweight='bold')
plt.show()
print(f'Matrice de mélange A :\n{A.round(3)}')

## 2. Algorithmes classiques

### 2a. Infomax (gradient batch)

In [ ]:
from ica import InfomaxICA

infomax = InfomaxICA(n_components=3, learning_rate=0.01, max_iter=500,
                     random_state=42, verbose=True)
S_infomax = infomax.fit_transform(X)
print(f'\nIterations: {infomax.n_iter_}')

### 2b. FastICA (from scratch)

In [ ]:
from ica import FastICACustom

fastica = FastICACustom(n_components=3, g='logcosh', random_state=42, verbose=True)
S_fastica = fastica.fit_transform(X)
print(f'\nIterations par composante: {fastica.n_iter_}')

## 3. Algorithmes stochastiques

### 3a. SGD-ICA

In [ ]:
from ica import SGDICA

sgd_ica = SGDICA(n_components=3, learning_rate=0.01, batch_size=64,
                 n_epochs=100, lr_schedule='cosine', random_state=42, verbose=True)
S_sgd = sgd_ica.fit_transform(X)

### 3b. Adam-ICA

In [ ]:
from ica import AdamICA

adam_ica = AdamICA(n_components=3, learning_rate=1e-3, batch_size=64,
                   n_epochs=100, random_state=42, verbose=True)
S_adam = adam_ica.fit_transform(X)

## 4. Benchmark complet et métriques

In [ ]:
from experiments.benchmark import run_benchmark

print('=== Benchmark ICA ===' )
results = run_benchmark(n_samples=3000, n_components=3, source_type='mixed',
                        random_state=42, verbose=True)

## 5. Visualisations

In [ ]:
from experiments.visualization import plot_sources, plot_convergence, plot_amari_scores

S_true = results['_meta']['S']

fig = plot_sources(S_true, results, n_show=400)
plt.show()

In [ ]:
fig = plot_convergence(results)
plt.show()

In [ ]:
fig = plot_amari_scores(results)
plt.show()

## 6. Scalabilité : SGD-ICA et Adam-ICA sur grands datasets

L'avantage principal des méthodes stochastiques est leur capacité à traiter
de grandes quantités de données sans charger tout le dataset en mémoire.

In [ ]:
import time

sizes = [1_000, 5_000, 20_000, 50_000]
times_sgd, times_adam, times_infomax = [], [], []

for n in sizes:
    _, _, X_large = make_sources(n_samples=n, n_sources=3, random_state=0)

    t0 = time.perf_counter()
    SGDICA(n_components=3, n_epochs=30, batch_size=128, random_state=0).fit(X_large)
    times_sgd.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    AdamICA(n_components=3, n_epochs=30, batch_size=128, random_state=0).fit(X_large)
    times_adam.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    InfomaxICA(n_components=3, max_iter=200, random_state=0).fit(X_large)
    times_infomax.append(time.perf_counter() - t0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, times_infomax, 'o-', label='Infomax (batch)', color='#e74c3c')
ax.plot(sizes, times_sgd, 's-', label='SGD-ICA', color='#3498db')
ax.plot(sizes, times_adam, '^-', label='Adam-ICA', color='#2ecc71')
ax.set_xlabel('Nombre d\'échantillons')
ax.set_ylabel('Temps (s)')
ax.set_title('Scalabilité : temps d\'entraînement vs. taille du dataset')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()